In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')


np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
#from conduct_evaluation import ConductEvaluation
#from normal_evaluation.drbart_evaluation import *
from PCR_evaluation.drbart_evaluation import *

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'pcr'

n_processes = 32
batch_size = 25

log_name = 'test'

with open('../transformed_event_logs/PCR_start_end_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

#test_event_log['time:timestamp'] = test_event_log['time:timestamp_complete']
test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_activities = ['Callback timeout', 'Export result', 'Export to EMS', 'Match patient data', 'Receive sample state', 'Send notification', 'Wait for plate validation', 'timeout']

In [3]:
N = 1000
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

likelihoods_A = None
likelihoods_A_S = None
likelihoods_A_S_AC = None
likelihoods_A_S_D = None
likelihoods_A_S_D_AC = None

In [4]:
drbart_model_A = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name/',
                     strict_parser=False)
evaluator_A = conduct_evaluation.ConductEvaluation(drbart_model_A, SampleOutcomes_DRBART_PCR_A, {
                                                        'activity_key' : 'concept:name',
                                                        #'resource_key' : 'org:resource_start',
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [5]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-0.2494923063879890772213724601')

In [6]:
np.mean(get_pscores(likelihoods_A))

np.float64(24488.52794873619)

In [7]:
drbart_model_A_S = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_seconds-in-day/',
                     strict_parser=False)
evaluator_A_S = conduct_evaluation.ConductEvaluation(drbart_model_A_S, SampleOutcomes_DRBART_PCR_A_S,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'categorical_args' : ['concept_name'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        #'known_activities' : known_activities,
                                                        #'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A_S = evaluator_A_S.sample_cases(False, True)

In [8]:
np.mean([v.ln() for v in likelihoods_A_S[0].values()])

Decimal('-143.8429267899433314663794524')

In [9]:
np.mean(get_pscores(likelihoods_A_S))

np.float64(12152.904985728346)

In [10]:
drbart_model_A_S_AC = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_seconds-in-day_activity-count/',
                     strict_parser=False)
evaluator_A_S_AC = conduct_evaluation.ConductEvaluation(drbart_model_A_S_AC, SampleOutcomes_DRBART_PCR_A_S_AC
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['concept_name',
                                                            '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        #'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A_S_AC = evaluator_A_S_AC.sample_cases(False, True)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1420499234.py, line 3)

In [11]:
np.mean([v.ln() for v in likelihoods_A_S_AC[0].values()])

TypeError: 'NoneType' object is not subscriptable

In [12]:
np.mean(get_pscores(likelihoods_A_S_AC))

TypeError: 'NoneType' object is not subscriptable

In [13]:
drbart_model_A_S_D = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_seconds-in-day_day-of-week/',
                     strict_parser=False)
evaluator_A_S_D = conduct_evaluation.ConductEvaluation(drbart_model_A_S_D, SampleOutcomes_DRBART_PCR_A_S_D,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['concept_name', 'day_of_week',
                                                                              #'(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        #'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A_S_D = evaluator_A_S_D.sample_cases(False, True)

In [14]:
np.mean([v.ln() for v in likelihoods_A_S_D[0].values()])

Decimal('-32.25412612447167837841203957')

In [15]:
np.mean(get_pscores(likelihoods_A_S_D))

np.float64(12053.68246145048)

In [16]:
results = {
    'drbart_model_A' : likelihoods_A,
    'drbart_model_R_A_S' : likelihoods_A_S,
    'drbart_model_R_A_S_AC' : likelihoods_A_S_AC,
    'drbart_model_R_A_S_D' : likelihoods_A_S_D,
}
with open('./'+model_name+'_dr_bart_evaluation_'+log_name+'.pickle', 'wb') as handle:
    pickle.dump(results, handle)